# Project 3: Name Classification
Anthony Josue Roman  
DATA 620: Web Analytics

## Introduction

For this project, I utilized the NLTK Names Corpus to categorize names based on their labels as either "male" or "female." As instructed in Chapter 6 of *Natural Language Processing with Python*, I will start with the simplest possible naive Bayes classifier using only the last letter of the names. Then, I will incrementally modify the feature set.

The data will be split into training, development-test, and test datasets. The training dataset will be used to train the classifiers, whereas the development-test dataset will be used to make improvements. The test dataset will be kept unused until the final model is chosen.

The objective is to find out if adding name attributes increases classification and to see if the chosen model produces similar results on both development-test and test datasets. These predictions represent labels within the corpus and do not imply defining one's gender identity.

## Setup and Reproducibility

NLTK is used to load the Names Corpus and create the baseline naive Bayes classifiers. The results are tabulated using Pandas, and the prediction evaluation and maximum entropy classifier using logistic regression are provided by scikit-learn.

The data split is reproducible because of the use of a fixed random seed. Each candidate uses the same training and development-test records, allowing consistent comparisons of feature and model changes. Only the selected model is evaluated on the final test set.

Below is the importation of the necessary libraries and loading of the Names Corpus if available.

In [ ]:
import sys
import random
from collections import Counter

import nltk
import pandas as pd
import sklearn

from nltk.corpus import names
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from IPython.display import display, Markdown

SEED = 620

try:
    names.words("male.txt")
    names.words("female.txt")
except LookupError:
    if not nltk.download("names", quiet=True):
        raise RuntimeError(
            "The Names Corpus could not be downloaded. "
            "Check your internet connection and rerun this cell."
        )

# Check packages for reproducibility.

print("Python:", sys.version.split()[0])
print("NLTK:", nltk.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("\nSetup complete. The Names Corpus is available.")

Python: 3.14.6
NLTK: 3.10.0
Pandas: 3.0.3
Scikit-learn: 1.9.0

Setup complete. The Names Corpus is available.


## Loading and Inspecting the Names Corpus

In The Names Corpus, there are two datasets of names categorized as either male or female. These datasets will be merged to form one dataset containing the pair of name-label as one observation. Names will be changed to lower case letters to avoid creating false variance from the spelling pattern due to capitalization.

Before splitting the data into training set and test set, I will evaluate the total number of observations, number of observations in each label and number of unique names. This is useful for detecting class imbalance and names which occur in both labels. Class imbalance is important because accuracy can be achieved by classifying the most frequent label.

Both names included under either label are retained as legitimate entries in the original corpus. Nevertheless, it implies that spelling cannot be relied upon exclusively for proper identification of the labels in the recordings.

There are 6,900 training entries, 500 development/test entries, and 500 test entries, making up a total of 7,900 entries. The typical NLTK corpus has 7,944 labeled entries. Once the count is checked, I will randomly choose 44 entries out of the way in order to maintain the exact split as required.

In [2]:
records = [
    (name.lower(), label)
    for label in ("male", "female")
    for name in names.words(f"{label}.txt")
]

assert all(name for name, label in records), "Empty names were found."
assert len(records) >= 7900, "Not enough records for the requested split."

label_counts = Counter(label for name, label in records)

corpus_summary = pd.DataFrame(
    label_counts.items(),
    columns=["Label", "Records"]
)
corpus_summary["Percentage"] = (
    corpus_summary["Records"] / len(records) * 100
).round(2)

display(corpus_summary)

unique_spellings = {name for name, label in records}

male_spellings = {
    name for name, label in records if label == "male"
}
female_spellings = {
    name for name, label in records if label == "female"
}
shared_spellings = male_spellings & female_spellings

print("Total labeled records:", len(records))
print("Unique name spellings:", len(unique_spellings))
print("Spellings listed under both labels:", len(shared_spellings))
print("Repeated identical name-label pairs:", len(records) - len(set(records)))
print("Records to set aside:", len(records) - 7900)

,Label,Records,Percentage
0,male,2943,37.05
1,female,5001,62.95


Total labeled records: 7944
Unique name spellings: 7576
Spellings listed under both labels: 365
Repeated identical name-label pairs: 3
Records to set aside: 44


## Splitting the Data

First, I randomize the record indices using the fixed seed and partition the data into 6,900 training records, 500 development-test records, and 500 test records. All other records are left untouched.

Training set is used for parameter estimation. The development-test set is used for comparison of features, error inspection and choosing a final classifier. Test set is held out from the rest of the data until the end of model selection.

Training and development-test sets are the same for all classifiers under consideration to allow for fair comparison of their results. Repeated model selection based on development-test results may tend to favor those patterns that are specific to this sample, so test set is required separately.

Finally, I verified that no record index appears in more than one subset. This does not remove duplicate name spellings or name-label pairs. Different records can still have the same name spelling, since the same name spelling may be associated with different labels. This is splitting labeled records, not completely unseen spellings.

In [3]:
indices = list(range(len(records)))
random.Random(SEED).shuffle(indices)

test_ids = indices[:500]
dev_ids = indices[500:1000]
train_ids = indices[1000:7900]
unused_ids = indices[7900:]

subsets = [set(train_ids), set(dev_ids), set(test_ids), set(unused_ids)]

assert sum(len(group) for group in subsets) == len(records)
assert len(set.union(*subsets)) == len(records)

train_names = [records[i] for i in train_ids]
dev_names = [records[i] for i in dev_ids]

assert len(train_names) == 6900
assert len(dev_names) == 500
assert len(test_ids) == 500

split_summary = pd.DataFrame({
    "Subset": ["Training", "Development-test", "Test", "Unused"],
    "Records": [
        len(train_names),
        len(dev_names),
        len(test_ids),
        len(unused_ids)
    ],
    "Purpose": [
        "Fit the classifiers",
        "Compare features and select the model",
        "Evaluate the selected model at the end",
        "Set aside to match the requested sizes"
    ]
})

display(split_summary)

development_counts = pd.DataFrame({
    "Training": Counter(label for name, label in train_names),
    "Development-test": Counter(label for name, label in dev_names)
}).rename_axis("Label")

display(development_counts)

print("Split verified: all record indices are disjoint.")

,Subset,Records,Purpose
0,Training,6900,Fit the classifiers
1,Development-test,500,Compare features and select the model
2,Test,500,Evaluate the selected model at the end
3,Unused,44,Set aside to match the requested sizes


,Training,Development-test
Label,,
female,4328,331
male,2572,169


Split verified: all record indices are disjoint.


## Establishing Baseline Performance

The number of records labeled as female in the training set is 4,328, whereas the number of records labeled as male is 2,572. In the development-test set, there are 331 records labeled as female and 169 labeled as male. From these figures, we can see that the two classes are not balanced.

Before training any classifiers, I created a baseline by predicting the most common class label, which is female in the training set. The classifier will correctly label 331 of the 500 records in the development-test set and will achieve an accuracy rate of 66.2% without making use of any spelling data.

I then train the naive Bayes classifier in chapter 6 using only the last letter of each name as features to get a learned baseline.

Accuracy is the ratio of correct prediction to all predictions. Both baselines use the training set to establish their predictions and are evaluated on the development-test set. The test set remains reserved for the final evaluation.

In [4]:
majority_label = Counter(
    label for name, label in train_names
).most_common(1)[0][0]

y_dev = [label for name, label in dev_names]
majority_predictions = [majority_label] * len(dev_names)
majority_accuracy = accuracy_score(y_dev, majority_predictions)

def last_letter(name):
    return {"suffix1": name[-1:]}

def make_features(rows, extractor):
    return [(extractor(name), label) for name, label in rows]


train_features = make_features(train_names, last_letter)

baseline_model = nltk.NaiveBayesClassifier.train(train_features)

baseline_train_accuracy = nltk.classify.accuracy(
    baseline_model, train_features
)

baseline_predictions = [
    baseline_model.classify(last_letter(name))
    for name, label in dev_names
]

baseline_dev_accuracy = accuracy_score(y_dev, baseline_predictions)

models = {
    "01 NB: last letter": (baseline_model, last_letter, "NLTK")
}

results = [{
    "Candidate": "01 NB: last letter",
    "Train accuracy": baseline_train_accuracy,
    "Dev accuracy": baseline_dev_accuracy,
    "Train minus dev": baseline_train_accuracy - baseline_dev_accuracy
}]

baseline_summary = pd.DataFrame({
    "Model": ["Majority-label baseline", "Naive Bayes: last letter"],
    "Dev-test accuracy (%)": [
        majority_accuracy * 100,
        baseline_dev_accuracy * 100
    ]
})

display(baseline_summary.round(2))

print(
    "Improvement over majority baseline:",
    f"{(baseline_dev_accuracy - majority_accuracy) * 100:.2f}",
    "percentage points"
)

,Model,Dev-test accuracy (%)
0,Majority-label baseline,66.2
1,Naive Bayes: last letter,76.0


Improvement over majority baseline: 9.80 percentage points


## Baseline Results and Error Analysis

The majority label baseline got an accuracy of 66.2% on the development-test data by getting 331 out of 500 records right. Naive Bayes using just the last character reached 76.0% by classifying 380 records. Thus, it resulted in an improvement by 9.8 percentage points or 49 extra correct classifications.

From the results we may conclude that the last character has some useful predictive power besides the ability to predict the most frequent label. Nevertheless, the classifier failed to classify 120 records, thus, this feature does not cover all the necessary spelling rules.

Prior to adding features, let us look at the mistakes to see what mistakes were made by the baseline. The following table will contain the name, the label, the prediction and the last character and characters of each name. More complex endings will allow differentiating the names having the same last character.

The error analysis will use only the development-test data. The test data will stay unseen to get the estimate after the model selection process.

In [5]:
baseline_errors = pd.DataFrame(
    [
        {
            "Name": name,
            "Actual": label,
            "Predicted": prediction,
            "Last letter": name[-1:],
            "Last two letters": name[-2:]
        }
        for (name, label), prediction
        in zip(dev_names, baseline_predictions)
        if label != prediction
    ],
    columns=[
        "Name", "Actual", "Predicted",
        "Last letter", "Last two letters"
    ]
)

print("Total development-test errors:", len(baseline_errors))

display(
    baseline_errors.sort_values("Name")
    .head(20)
    .reset_index(drop=True)
)

errors_by_ending = (
    baseline_errors.groupby("Last letter")
    .size()
    .sort_values(ascending=False)
    .rename("Error count")
    .to_frame()
)

display(errors_by_ending.head(10))

Total development-test errors: 120


,Name,Actual,Predicted,Last letter,Last two letters
0,abbie,male,female,e,ie
1,abigael,female,male,l,el
2,andre,male,female,e,re
3,archy,male,female,y,hy
4,ardelis,female,male,s,is
5,arlyn,female,male,n,yn
6,austin,female,male,n,in
7,bartholemy,male,female,y,my
8,bealle,male,female,e,le
9,bird,female,male,d,rd


,Error count
Last letter,
n,36
e,25
y,14
l,12
h,7
s,5
t,5
o,4
i,3


## Interpreting Errors and Adding a Two-Letter Suffix

The baseline had an error count of 36 for names with the n ending, 25 for those with the e ending, and 14 for names ending in y. This is a count and not an error rate; therefore, it does not determine which ending is the most difficult to label since the count must be adjusted by how frequently each ending occurs.

The examples provide insights that cannot be seen with the single letter feature. For example, arlyn and carlyn are categorized as female in the corpus data but classified as male by the baseline. This is because both of them end with yn, a letter that the baseline cannot discriminate from other names ending in n. Also, andre and blaine are labeled as males in the corpus but predicted as female since only their final e was taken into consideration.

In the subsequent experiment, I kept the single-letter feature and the final two letters of each name. This means that the classifier will get more detailed spelling information from just a change in the feature set.

I trained another naive Bayes classifier using the same training records and test it using the same development-test records.

In [6]:
def suffix_two(name):
    return {
        "suffix1": name[-1:],
        "suffix2": name[-2:]
    }

def evaluate_nb(key, extractor):
    training_features = make_features(train_names, extractor)
    model = nltk.NaiveBayesClassifier.train(training_features)

    train_accuracy = nltk.classify.accuracy(
        model, training_features
    )

    predictions = [
        model.classify(extractor(name))
        for name, label in dev_names
    ]
    dev_accuracy = accuracy_score(y_dev, predictions)

    models[key] = (model, extractor, "NLTK")

    results[:] = [
        row for row in results if row["Candidate"] != key
    ]
    results.append({
        "Candidate": key,
        "Train accuracy": train_accuracy,
        "Dev accuracy": dev_accuracy,
        "Train minus dev": train_accuracy - dev_accuracy
    })

    print(f"Training accuracy: {train_accuracy:.2%}")
    print(f"Development-test accuracy: {dev_accuracy:.2%}")
    print(
        "Change from last-letter baseline:",
        f"{(dev_accuracy - baseline_dev_accuracy) * 100:+.2f}",
        "percentage points"
    )

    return predictions


suffix_two_predictions = evaluate_nb(
    "02 NB: suffixes 1-2",
    suffix_two
)

display(
    pd.DataFrame(results)[["Candidate", "Dev accuracy"]]
    .assign(**{
        "Dev accuracy": lambda df: df["Dev accuracy"] * 100
    })
    .rename(columns={"Dev accuracy": "Dev-test accuracy (%)"})
    .round(2)
)

Training accuracy: 79.04%
Development-test accuracy: 78.20%
Change from last-letter baseline: +2.20 percentage points


,Candidate,Dev-test accuracy (%)
0,01 NB: last letter,76.0
1,02 NB: suffixes 1-2,78.2


## Two-Letter Suffix Results and the Next Experiment

The use of the two-letter suffix improved development-test accuracy by 2.2 percentage points from 76.0% to 78.2%. The model was able to categorize 391 of 500 cases, against 380 for the last letter suffix.

Accuracy of the model in training was 79.04%, which is around 0.84 percentage points higher than the development-test accuracy. This narrow gap does not imply that there is significant loss of accuracy between the two sets of data, but it does not show how well the model would work on another set of data.

This improvement shows the usefulness of adding more information about the suffixes in names. In the following experiment, the one-letter and the two-letter suffixes would be used along with the three-letter suffix.

The longer suffixes can be more specific and thus can appear rarely in the training set. Hence, it would make sense to test not only the accuracy in training but also the development-test accuracy.

In [7]:
def suffix_three(name):
    return {
        **suffix_two(name),
        "suffix3": name[-3:]
    }

suffix_three_predictions = evaluate_nb(
    "03 NB: suffixes 1-3",
    suffix_three
)

previous_accuracy = accuracy_score(y_dev, suffix_two_predictions)
current_accuracy = accuracy_score(y_dev, suffix_three_predictions)

print(
    "Change from two-letter suffix model:",
    f"{(current_accuracy - previous_accuracy) * 100:+.2f}",
    "percentage points"
)

comparison = pd.DataFrame(results).copy()

for column in ["Train accuracy", "Dev accuracy", "Train minus dev"]:
    comparison[column] *= 100

comparison = comparison.rename(columns={
    "Train accuracy": "Training accuracy (%)",
    "Dev accuracy": "Dev-test accuracy (%)",
    "Train minus dev": "Train-dev gap (pp)"
})

display(comparison.round(2))

Training accuracy: 82.96%
Development-test accuracy: 79.60%
Change from last-letter baseline: +3.60 percentage points
Change from two-letter suffix model: +1.40 percentage points


,Candidate,Training accuracy (%),Dev-test accuracy (%),Train-dev gap (pp)
0,01 NB: last letter,76.42,76.0,0.42
1,02 NB: suffixes 1-2,79.04,78.2,0.84
2,03 NB: suffixes 1-3,82.96,79.6,3.36


## Three-Letter Suffix Results and Adding Prefixes

The addition of the suffix has improved the development test accuracy of 78.2% to 79.6%, with an improvement of 1.4 percentage points. The model managed to classify 398 out of 500 observations, leaving 102 errors. Compared to the baseline last letter model, there has been an improvement of 3.6 percentage points in terms of accuracy.

Training accuracy improved to 82.96%, generating a training and development difference of 3.36 percentage points. It seems that the increase in training and development differences, compared to 0.84 percentage points in the previous model, indicates that there might be some non-generalizable patterns discovered during training.

Up until now, all the selected features are related to the endings of names. For the following experiment, I keep all three ending features and include the first letter and first two letters of the name. These letters that start the name are called prefixes and can carry information that complements the ending of names.

In [8]:
def affixes(name):
    return {
        **suffix_three(name),
        "prefix1": name[:1],
        "prefix2": name[:2]
    }

affix_predictions = evaluate_nb(
    "04 NB: prefixes and suffixes",
    affixes
)

previous_accuracy = accuracy_score(y_dev, suffix_three_predictions)
current_accuracy = accuracy_score(y_dev, affix_predictions)

print(
    "Change from three-letter suffix model:",
    f"{(current_accuracy - previous_accuracy) * 100:+.2f}",
    "percentage points"
)

comparison = pd.DataFrame(results).copy()

for column in ["Train accuracy", "Dev accuracy", "Train minus dev"]:
    comparison[column] *= 100

comparison = comparison.rename(columns={
    "Train accuracy": "Training accuracy (%)",
    "Dev accuracy": "Dev-test accuracy (%)",
    "Train minus dev": "Train-dev gap (pp)"
})

display(comparison.round(2))

Training accuracy: 84.83%
Development-test accuracy: 82.20%
Change from last-letter baseline: +6.20 percentage points
Change from three-letter suffix model: +2.60 percentage points


,Candidate,Training accuracy (%),Dev-test accuracy (%),Train-dev gap (pp)
0,01 NB: last letter,76.42,76.0,0.42
1,02 NB: suffixes 1-2,79.04,78.2,0.84
2,03 NB: suffixes 1-3,82.96,79.6,3.36
3,04 NB: prefixes and suffixes,84.83,82.2,2.63


## Prefix Results and Adding Name Structure

The addition of the first letter and two-letter prefix raised development-test accuracy to 82.2%, which is 2.6 percentage points higher compared to the 79.6%. The model has been able to categorize 411 of 500 records, hence committing only 89 errors. This development-test accuracy is the highest amongst the models tested so far.

The training accuracy of this model is 84.83%, 2.63 percentage points higher than the development-test accuracy. The gap between the training and development-test accuracy is smaller compared to the 3.36 point difference observed for the suffix-only model. In this case, the prefix was able to improve the development accuracy and reduce the gap.

For the next experiment, I maintained the features of prefix and suffix and add name length, vowel count, and the presence or absence of adjacent repeated letters. These are features that describe the name structure.

In this context, the definition of vowels is a, e, i, o, and u. This is based on spelling and not pronunciation. Name length and vowels count are encoded as categorical while adjacent repeated letters are encoded as True or False.

These additional features do not have to improve the classification process.

In [9]:
def structure(name):
    return {
        **affixes(name),
        "length": str(len(name)),
        "vowel_count": str(sum(letter in "aeiou" for letter in name)),
        "double_letter": any(
            first == second
            for first, second in zip(name, name[1:])
        )
    }

structure_predictions = evaluate_nb(
    "05 NB: affixes and structure",
    structure
)

previous_accuracy = accuracy_score(y_dev, affix_predictions)
current_accuracy = accuracy_score(y_dev, structure_predictions)

print(
    "Change from prefix-and-suffix model:",
    f"{(current_accuracy - previous_accuracy) * 100:+.2f}",
    "percentage points"
)

comparison = pd.DataFrame(results).copy()

for column in ["Train accuracy", "Dev accuracy", "Train minus dev"]:
    comparison[column] *= 100

comparison = comparison.rename(columns={
    "Train accuracy": "Training accuracy (%)",
    "Dev accuracy": "Dev-test accuracy (%)",
    "Train minus dev": "Train-dev gap (pp)"
})

display(comparison.round(2))

Training accuracy: 84.30%
Development-test accuracy: 81.20%
Change from last-letter baseline: +5.20 percentage points
Change from prefix-and-suffix model: -1.00 percentage points


,Candidate,Training accuracy (%),Dev-test accuracy (%),Train-dev gap (pp)
0,01 NB: last letter,76.42,76.0,0.42
1,02 NB: suffixes 1-2,79.04,78.2,0.84
2,03 NB: suffixes 1-3,82.96,79.6,3.36
3,04 NB: prefixes and suffixes,84.83,82.2,2.63
4,05 NB: affixes and structure,84.30,81.2,3.10


## Structure Feature Results and Maximum Entropy Classification

Using the three new features of name length, number of vowels, and consecutive letters lowers the development set accuracy from 82.2% to 81.2%. The extended model is able to classify 406 out of 500 samples, resulting in 94 mistakes. Training accuracy is 84.30%, and there is a difference of about 3.10 percentage points between training and development sets.

The prefix and suffix-based model still seems to be the best naive Bayes classifier model. The decrease proves that adding more features is not necessarily improving the algorithm. The new features can have weak or redundant data, but the current analysis does not prove this exactly.

Conditional independence is an important feature of naive Bayes. It does not hold true when talking about overlapping suffixes and related features like name length and vowel count.

The next classifier to be tested in my experiments is maximum entropy classification, which is one of the classifier classes covered in Chapter 6. I will do this using scikit-learn's logistic regression class, which is a binary maximum entropy classifier. In this model, all feature weights are learned simultaneously without the conditional independence assumption.

The same feature set used in the previous experiment is employed here to see how these classifiers cope with the information in this feature set. Here, DictVectorizer will transform the feature dictionaries into columns and learn them from training examples only.

This model employs L2 regularization, which penalizes large values of feature weights. For the regularization parameter, C is fixed to 1.0, and performance is evaluated on the development-test set.

In [10]:
X_train = [structure(name) for name, label in train_names]
y_train = [label for name, label in train_names]
X_dev = [structure(name) for name, label in dev_names]

maxent = make_pipeline(
    DictVectorizer(),
    LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        random_state=SEED
    )
)

maxent.fit(X_train, y_train)

assert maxent[-1].n_iter_.max() < 1000, (
    "The optimizer reached its iteration limit. "
    "Check convergence before interpreting the results."
)

maxent_train_accuracy = accuracy_score(
    y_train, maxent.predict(X_train)
)
maxent_predictions = maxent.predict(X_dev)
maxent_dev_accuracy = accuracy_score(y_dev, maxent_predictions)

key = "06 MaxEnt: affixes and structure"
models[key] = (maxent, structure, "sklearn")

results[:] = [row for row in results if row["Candidate"] != key]
results.append({
    "Candidate": key,
    "Train accuracy": maxent_train_accuracy,
    "Dev accuracy": maxent_dev_accuracy,
    "Train minus dev": maxent_train_accuracy - maxent_dev_accuracy
})

print(f"Training accuracy: {maxent_train_accuracy:.2%}")
print(f"Development-test accuracy: {maxent_dev_accuracy:.2%}")

previous_accuracy = accuracy_score(y_dev, structure_predictions)
print(
    "Change from naive Bayes with the same features:",
    f"{(maxent_dev_accuracy - previous_accuracy) * 100:+.2f}",
    "percentage points"
)

comparison = pd.DataFrame(results).copy()

for column in ["Train accuracy", "Dev accuracy", "Train minus dev"]:
    comparison[column] *= 100

comparison = comparison.rename(columns={
    "Train accuracy": "Training accuracy (%)",
    "Dev accuracy": "Dev-test accuracy (%)",
    "Train minus dev": "Train-dev gap (pp)"
})

display(comparison.round(2))

Training accuracy: 86.65%
Development-test accuracy: 84.40%
Change from naive Bayes with the same features: +3.20 percentage points


,Candidate,Training accuracy (%),Dev-test accuracy (%),Train-dev gap (pp)
0,01 NB: last letter,76.42,76.0,0.42
1,02 NB: suffixes 1-2,79.04,78.2,0.84
2,03 NB: suffixes 1-3,82.96,79.6,3.36
3,04 NB: prefixes and suffixes,84.83,82.2,2.63
4,05 NB: affixes and structure,84.30,81.2,3.10
5,06 MaxEnt: affixes and structure,86.65,84.4,2.25


## Maximum Entropy Results and Final Model Selection

Maximum Entropy obtained an accuracy of 84.4% on the development test set, making 422 correct predictions out of 500 and committing 78 errors. It beats the naive Bayes algorithm with the same feature set by 3.2 percentage points and the best naive Bayes model by 2.2 percentage points.

In comparison with the original classifier based on the last letter, development-test accuracy increased by 8.4 percentage points, for an improvement of 42 predictions. Training accuracy is 86.65%, creating a training-development difference of about 2.25 percentage points.

For model selection, I pick the one with the highest development-test accuracy. In case of a tie between multiple models, I prefer the earlier experiment. According to this criterion, the maximum entropy classifier with prefix, suffix, and structure features becomes the selected classifier. It is the best model among those tested, but not necessarily the best name classifier.

Development of the model is now finished. I retain the selected model's features, parameters and configuration. I do not train it again using the training set and the development test set together, so the development and test accuracies characterize the same model.

The test set will be used to estimate the performance of the chosen model on unseen data. Since development-test performance was used to make the decision about model choice, this performance figure can be considered to be too optimistic. The test set accuracy can be either lower or higher depending on the composition of the sample of names.

In [11]:
selection_table = (
    pd.DataFrame(results)
    .sort_values("Candidate")
    .reset_index(drop=True)
)

best_index = selection_table["Dev accuracy"].idxmax()
best_key = selection_table.loc[best_index, "Candidate"]

final_model, final_extractor, implementation = models[best_key]
selected_dev_accuracy = float(
    selection_table.loc[best_index, "Dev accuracy"]
)

def predict_rows(model, extractor, engine, rows):
    features = [extractor(name) for name, label in rows]

    if engine == "sklearn":
        return list(model.predict(features))

    return model.classify_many(features)

final_dev_predictions = predict_rows(
    final_model, final_extractor, implementation, dev_names
)

assert (
    accuracy_score(y_dev, final_dev_predictions)
    == selected_dev_accuracy
)

test_names = [records[i] for i in test_ids]
y_test = [label for name, label in test_names]

test_predictions = predict_rows(
    final_model, final_extractor, implementation, test_names
)
test_accuracy = accuracy_score(y_test, test_predictions)

print("Selected model:", best_key)

final_summary = pd.DataFrame({
    "Subset": ["Development-test", "Test"],
    "Records": [len(dev_names), len(test_names)],
    "Correct predictions": [
        sum(actual == predicted
            for actual, predicted in zip(y_dev, final_dev_predictions)),
        sum(actual == predicted
            for actual, predicted in zip(y_test, test_predictions))
    ],
    "Accuracy (%)": [
        selected_dev_accuracy * 100,
        test_accuracy * 100
    ]
})

display(final_summary.round(2))

print(
    "Test minus development-test accuracy:",
    f"{(test_accuracy - selected_dev_accuracy) * 100:+.2f}",
    "percentage points"
)

Selected model: 06 MaxEnt: affixes and structure


,Subset,Records,Correct predictions,Accuracy (%)
0,Development-test,500,422,84.4
1,Test,500,392,78.4


Test minus development-test accuracy: -6.00 percentage points


## Comparing Test and Development-Test Performance

The maximum entropy classifier that was selected classified 392 out of 500 test records, thus achieving 78.4% accuracy. In the case of the development-test set, 422 out of 500 records were classified, which resulted in 84.4% accuracy. Test accuracy, therefore, was six percentage points lower, and there were 108 mistakes instead of 78 on the development-test set.

Some difference in scores is expected because there are different sets of names included. Besides, the development-test set determined the feature set and the classifier selected, which makes its score an optimistic estimate for the performance on new records.

A difference between the scores is expected, but a decrease of exactly six percentage points was not guaranteed. Sampling variation and selection on the development-test set are plausible contributors, but this single split cannot establish their individual effects.

Final held-out estimate of the performance is 78.4%. The model selected is retained as it is, no further tuning is done after seeing the test set scores. Otherwise, the test set will not function as a final evaluation.

## Examining Performance by Label

Accuracy does not give an indication of whether the performance of the model is consistent across both corpus labels. I will then investigate the confusion matrix and classification report using the current test results.

F1 is the harmonic mean of precision and recall. Support is the number of test records belonging to each label. The confusion matrix uses rows for actual labels and columns for predicted labels. The diagonal elements contain the correct predictions, and the non-diagonal elements contain errors.

The precision score is used to determine how often the prediction of a label is correct. The recall score shows the ratio of the records whose actual label is that label and which are predicted correctly by the model.

In [12]:
labels = ["female", "male"]

confusion_table = pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=labels),
    index=pd.Index(labels, name="Actual label"),
    columns=pd.Index(labels, name="Predicted label")
)

print("Test-set confusion matrix:")
display(confusion_table)

print("\nTest-set classification report:")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=labels,
        digits=3,
        zero_division=0
    )
)

correct_predictions = sum(
    confusion_table.loc[label, label] for label in labels
)

assert confusion_table.to_numpy().sum() == len(y_test)
assert correct_predictions == sum(
    actual == predicted
    for actual, predicted in zip(y_test, test_predictions)
)

print(f"Correct predictions: {correct_predictions} / {len(y_test)}")
print(f"Test accuracy: {correct_predictions / len(y_test):.2%}")

Test-set confusion matrix:


Predicted label,female,male
Actual label,,
female,261,55
male,53,131



Test-set classification report:
              precision    recall  f1-score   support

      female      0.831     0.826     0.829       316
        male      0.704     0.712     0.708       184

    accuracy                          0.784       500
   macro avg      0.768     0.769     0.768       500
weighted avg      0.785     0.784     0.784       500

Correct predictions: 392 / 500
Test accuracy: 78.40%


## Interpreting Performance by Label

From the confusion matrix, it can be seen that there were 261 out of 316 female-labeled records classified correctly and 131 out of 184 male-labeled records classified correctly. There were 55 female-labeled records classified as male, and 53 male-labeled records classified as female. Hence, there were 392 correct classifications and 108 errors.

While there are nearly an equal number of errors, the classes have different sizes. For the females, the recall was 82.6%, whereas for males, it was 71.2%. In this regard, the model made mistakes for a greater percentage of the males.

For the females, precision was 83.1% as compared to 70.4% for the male label. This translated to 0.829 and 0.708 F1 scores respectively. These values are indicative of better performance of the model for the female label in all three measures.

The macro-average F1 score was 0.768, whereas the weighted-average F1 score was 0.784. As is clear, the latter is greater since it gives more weight to the class with better performance (the female class).

## Checking Name Overlap Across Subsets

This approach avoids the possibility of the same record occurring in both the training and test sets, although different records may have the same spelling of the name. Such a situation may arise where the name is classified into both corpora.

I then examined the number of test records that have a spelling of the name that also occurs in the training set and assess the accuracy of these against records that had never been seen before. This is a descriptive exercise carried out on the current predictions from the test set.

In [13]:
training_spellings = {name for name, label in train_names}

overlap_details = pd.DataFrame({
    "Name": [name for name, label in test_names],
    "Actual": y_test,
    "Predicted": test_predictions,
    "Seen in training": [
        name in training_spellings for name, label in test_names
    ]
})

overlap_details["Correct"] = (
    overlap_details["Actual"] == overlap_details["Predicted"]
)

overlap_rows = []

for seen, description in [
    (True, "Seen in training"),
    (False, "Unseen in training")
]:
    group = overlap_details[
        overlap_details["Seen in training"] == seen
    ]

    overlap_rows.append({
        "Test spelling": description,
        "Records": len(group),
        "Correct predictions": int(group["Correct"].sum()),
        "Accuracy (%)": (
            group["Correct"].mean() * 100
            if len(group) > 0 else float("nan")
        )
    })

overlap_summary = pd.DataFrame(overlap_rows)
display(overlap_summary.round(2))

assert overlap_summary["Records"].sum() == len(test_names)
assert overlap_summary["Correct predictions"].sum() == sum(
    actual == predicted
    for actual, predicted in zip(y_test, test_predictions)
)

print(
    "Unique test spellings also present in training:",
    overlap_details.loc[
        overlap_details["Seen in training"], "Name"
    ].nunique()
)

,Test spelling,Records,Correct predictions,Accuracy (%)
0,Seen in training,43,20,46.51
1,Unseen in training,457,372,81.40


Unique test spellings also present in training: 43


## Interpreting Spelling Overlap and Study Limitations

Among the 500 test cases, 43 were overlapping spellings and 457 were unseen spellings. The classifier correctly classified 20 out of 43 overlapping cases, with an accuracy rate of 46.51%. For unseen spellings, the classifier correctly classified 372 out of 457 cases, with an accuracy rate of 81.40%.

It seems likely that the lower accuracy rate for overlapping spellings results from names being listed under both corpus labels. Spellings that occurred with one label in training may occur with the other label in the test set. This table only checks the spellings and not label overlaps for each individual record.

These sets have differences in terms of content and number of records; therefore, there is no proof of poor accuracy rates caused by seeing the name in training. Also, the overlapping set is quite small.

The overall accuracy of the test still holds at 78.4%. The 81.40% accuracy of unseen spellings refers to a descriptive analysis of a subgroup of results, and does not replace the overall test accuracy.

The present study makes use of one fixed split and six candidate models. Multiple comparisons using the development-test set might be biased by particular patterns from the subset. The data set used in this study does not have equal classes, and might not represent patterns of names in all cultures or eras.

Further research could include cross-validation on the development set and a grouped split that keeps all records with the same spelling in one subset. Such an approach would require a completely different test design.

## Conclusion

This analysis explored the potential of further refinement of the name classifier beyond that of the last-letter case in Chapter 6 through incremental feature engineering. Development-test accuracy of the naive Bayes classifier improved from 76.0% to 82.2% with the inclusion of more extended suffix and prefix features. The addition of name features reduced the accuracy of naive Bayes to 81.2%, showing that sometimes additional features may not be helpful.

Maximum entropy classification using the extended feature set reached the highest development-test accuracy at 84.4%. This was an 8.4 percentage point improvement over the initial classifier's accuracy.

The chosen classifier had an accuracy of 78.4% on the holdout test set, correctly predicting 392 of 500 cases. This was 6.0 percentage points lower than its development-test accuracy. Some variation is to be expected since the two sets have different names and development-test performance drove model choice. However, the exact nature of this discrepancy could not be known beforehand.

Additionally, the classification report indicated higher precision for the category of females compared to males. The spelling-overlap comparison identified yet another problem: distinct entries may have common spellings, and a given name can be linked to both categories.

In summary, the experiments illustrate the utility of incremental comparison of features, error analysis, and a dedicated final test set. The final performance result is 78.4% test accuracy. The model classifies historical categories within this dataset and does not assign any gender identity.

## References

Bird, S., Klein, E., & Loper, E. (2009). *Natural language processing
with Python*. O'Reilly Media. Chapter 6: Learning to classify text.
https://www.nltk.org/book/ch06.html

NLTK Project. (n.d.). *Names Corpus*. Accessed through
`nltk.corpus.names`, using the `male.txt` and `female.txt` files.

Scikit-learn developers. (n.d.). *LogisticRegression*.
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

Scikit-learn developers. (n.d.). *DictVectorizer*.
https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.DictVectorizer.html
